# Data Evaluation

In [1]:
import os
from pathlib import Path

import boto3
import evidently.metrics as em
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataDriftPreset, DatasetStats, DataSummaryPreset

In [2]:
PROJ_ROOT = Path.cwd().parent

In [3]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

In [4]:
import r2.io_utils as r2io

## About

Ensuring the integrity and stability of the underlying data is just as critical as the predictive performance of a ML model. Running data drift, data quality, and descriptive statistics checks between our train, validation, and test splits helps verify that the model is being evaluated under conditions consistent with what it learned. Column-level drift detection can reveal shifts in feature distributions (e.g. customer spending behavior or account activity), while dataset- and column-level quality checks can surface issues such as missing values, outliers, or schema inconsistencies that may silently degrade model performance. Together, these tests help place the model results into an appropriate context when we make conclusions from exploratory data analysis and from the model validation and model evaluation stages of model development. Ultimately, this leads to more reliable model evaluation and predictions, and also better-informed business decisions regarding customer retention.

So, the following data tests are run using the [Evidently AI](https://www.evidentlyai.com/) [Python package](https://pypi.org/project/evidently/)

1. Data drift at the [column level](https://docs.evidentlyai.com/metrics/all_metrics#data-drift)
2. Data quality
   - [dataset level](https://docs.evidentlyai.com/metrics/all_metrics#dataset-data-quality)
   - [column level](https://docs.evidentlyai.com/metrics/all_metrics#column-data-quality)
3. Descriptive Statistics at the [column level](https://docs.evidentlyai.com/metrics/all_metrics#value-stats)

### Outputs

None

## User Inputs

In [5]:
# R2 data bucket details
bucket_name = "cc-churn-splits"
# # name of train data key (file) in private R2 bucket
r2_key_train = "train_data.parquet.gzip"
# # name of validation data key (file) in private R2 bucket
r2_key_val = "validation_data.parquet.gzip"
# # name of test data key (file) in private R2 bucket
r2_key_test = "test_data.parquet.gzip"

# datatypes for categorical columns
dtypes_categoricals = {
    "gender": "string[pyarrow]",
    "marital_status": "string[pyarrow]",
    "income_category": "string[pyarrow]",
    "card_category": "string[pyarrow]",
    "education_level": "string[pyarrow]",
}

ordinal_features = [
    "income_category",
    "education_level",
]
categorical_features = [
    "card_category",
    "marital_status",
]
numeric_features = [
    "customer_age",
    "dependent_count",
    "months_on_book",
    "num_products",
    "months_inactive_12_mon",
    "contacts_count_12_mon",
    "total_revolv_bal",
    "avg_open_to_buy",
    "total_amt_chng_q4_q1",
    "total_trans_amt",
    "total_trans_ct",
    "total_ct_chng_q4_q1",
    "avg_utilization_ratio",
]
features = numeric_features + ordinal_features  # +categorical_features

In [6]:
account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID_USER2")
secret_access_key = os.getenv("SECRET_ACCESS_KEY_USER2")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

Define a function to extract the expected lower and upper bounds of the range in which the actual values (or proportions) in the current dataset should fall

In [7]:
def extract_expected_bounds(df: pd.DataFrame) -> pd.DataFrame:
    pattern = (
        r".*:\s+Actual value ([\d.]+)(?:\s+,\s+but\s+|\s{2,})expected\s+"
        r"([\d.]+)\s+±\s+([\d.]+)"
    )
    df = df.assign(
        actual=lambda x: x["description"]
        .str.extract(pattern)[0]
        .astype(float),
        expected_min=lambda x: x["description"]
        .str.extract(pattern)
        .astype(float)
        .pipe(lambda d: d[1] - d[2]),
        expected=lambda x: x["description"]
        .str.extract(pattern)[1]
        .astype(float),
        expected_max=lambda x: x["description"]
        .str.extract(pattern)
        .astype(float)
        .pipe(lambda d: d[1] + d[2]),
    )
    return df

Define a function to run data tests relative to a reference dataset, at the column level

In [8]:
def run_column_data_tests(
    metrics, dataset_test, dataset_train
) -> pd.DataFrame:
    report = Report(metrics, include_tests=True)
    my_eval = report.run(dataset_test, dataset_train)
    eval_dict = my_eval.dict()

    df_metrics = pd.DataFrame.from_records(
        [
            {
                "id": r["id"],
                "column": r["config"]["column"],
                "metric_name": r["metric_name"].split("(")[0],
                "value": (
                    r["value"]["count"]
                    if isinstance(r["value"], dict)
                    else r["value"]
                ),
                "threshold": (
                    r["config"]["threshold"]
                    if "threshold" in list(r["config"])
                    else np.nan
                ),
            }
            for r in eval_dict["metrics"]
        ]
    )
    df_tests = pd.DataFrame.from_records(
        [
            {
                "condition": test["id"],
                "column": test["metric_config"]["params"]["column"],
                "category": (
                    test["metric_config"]["params"]["categories"]
                    if "categories" in list(test["metric_config"]["params"])
                    else []
                ),
                "id": test["metric_config"]["metric_id"],
                "metric_name": test["metric_config"]["params"]["type"].split(
                    ":"
                )[-1],
                "is_count": (
                    test["bound_test"]["is_count"]
                    if "is_count" in list(test["bound_test"])
                    else True
                ),
                "description": test["description"],
                "tolerance": (
                    test["test_config"]["expected"]["relative"]
                    if "expected" in list(test["test_config"])
                    else np.nan
                ),
                "status": test["status"].value,
            }
            for test in eval_dict["tests"]
        ]
    )
    df_summary = df_metrics.merge(
        df_tests.query(
            "((category.str.len() == 0) & (is_count == True)) | "
            "((category.str.len() >= 1) & (is_count == False))"
        ),
        on=["column", "id", "metric_name"],
        how="left",
    )
    return df_summary

Define a function to run data tests relative to a reference dataset, at the dataset level

In [9]:
def run_dataset_data_tests(
    metrics, dataset_val, dataset_train
) -> pd.DataFrame:
    report = Report(metrics, include_tests="True")
    my_eval = report.run(dataset_test, dataset_train)
    eval_dict = my_eval.dict()

    df_metrics = pd.DataFrame.from_records(
        [
            {
                "id": r["id"],
                "metric_name": r["metric_name"].split("(")[0],
                "value": (
                    r["value"]["count"]
                    if isinstance(r["value"], dict)
                    else r["value"]
                ),
            }
            for r in eval_dict["metrics"]
        ]
    )
    df_tests = pd.DataFrame.from_records(
        [
            {
                "condition": test["id"],
                "metric_name": test["metric_config"]["params"]["type"].split(
                    ":"
                )[-1],
                "description": test["description"],
                "status": test["status"].value,
            }
            for test in eval_dict["tests"]
        ]
    )
    df_summary = df_metrics.merge(df_tests, on=["metric_name"])
    return df_summary

## Load Data

### Data for Model Validation

Load the training data

In [10]:
%%time
df_train = (
    r2io.pandas_read_parquet_r2(s3_client, bucket_name, r2_key_train)
    .astype({c: float for c in numeric_features})
    .astype({c: str for c in categorical_features+ordinal_features})
)
print(f"Loaded {len(df_train):,} rows of training data")
with pd.option_context('display.max_columns', None):
    display(df_train.head())

Loaded 6,285 rows of training data


,clientnum,is_churned,customer_age,gender,dependent_count,education_level,marital_status,income_category,card_category,months_on_book,num_products,months_inactive_12_mon,contacts_count_12_mon,credit_limit,total_revolv_bal,avg_open_to_buy,total_amt_chng_q4_q1,total_trans_amt,total_trans_ct,total_ct_chng_q4_q1,avg_utilization_ratio
0,714283458,0,40.0,M,2.0,College,Single,$80K - $120K,Blue,36.0,5.0,1.0,4.0,14544.0,0.0,14544.0,0.768,4064.0,92.0,0.769,0.000
1,787587033,0,42.0,F,4.0,Graduate,Single,Less than $40K,Blue,32.0,3.0,1.0,2.0,2996.0,1992.0,1004.0,0.948,4463.0,87.0,0.740,0.665
2,714672933,0,52.0,F,3.0,Graduate,Married,$40K - $60K,Blue,36.0,6.0,4.0,2.0,3143.0,2268.0,875.0,0.801,4417.0,84.0,0.680,0.722
3,714974658,0,48.0,F,4.0,College,Married,Less than $40K,Blue,36.0,6.0,1.0,4.0,2464.0,1867.0,597.0,0.600,1219.0,35.0,1.333,0.758
4,712049208,0,56.0,M,3.0,Post-Graduate,Married,$60K - $80K,Blue,39.0,3.0,1.0,2.0,3955.0,2517.0,1438.0,0.484,1238.0,25.0,1.083,0.636


CPU times: user 102 ms, sys: 16.8 ms, total: 119 ms
Wall time: 356 ms


### Data for Model Evaluation

Load the validation data

In [11]:
%%time
df_val = (
    r2io.pandas_read_parquet_r2(s3_client, bucket_name, r2_key_val)
    .astype({c: float for c in numeric_features})
    .astype({c: str for c in categorical_features+ordinal_features})
)
print(f"Loaded {len(df_val):,} rows of validation data")
with pd.option_context('display.max_columns', None):
    display(df_val.head())

Loaded 1,693 rows of validation data


,clientnum,is_churned,customer_age,gender,dependent_count,education_level,marital_status,income_category,card_category,months_on_book,num_products,months_inactive_12_mon,contacts_count_12_mon,credit_limit,total_revolv_bal,avg_open_to_buy,total_amt_chng_q4_q1,total_trans_amt,total_trans_ct,total_ct_chng_q4_q1,avg_utilization_ratio
0,823840458,0,47.0,F,1.0,High School,Married,Unknown,Blue,43.0,4.0,2.0,2.0,1828.0,1517.0,311.0,0.661,4542.0,82.0,0.577,0.830
1,716328558,0,46.0,M,2.0,Uneducated,Married,$80K - $120K,Blue,38.0,4.0,5.0,2.0,11434.0,0.0,11434.0,0.840,4520.0,83.0,0.844,0.000
2,714735033,0,45.0,F,5.0,Doctorate,Married,Less than $40K,Blue,34.0,4.0,2.0,2.0,1438.3,491.0,947.3,0.708,4376.0,84.0,0.787,0.341
3,712163433,1,47.0,M,3.0,Uneducated,Married,$40K - $60K,Blue,29.0,1.0,2.0,4.0,1684.0,644.0,1040.0,0.723,2164.0,36.0,0.714,0.382
4,720307683,0,43.0,F,4.0,Graduate,Divorced,Less than $40K,Blue,36.0,3.0,2.0,2.0,1438.3,743.0,695.3,0.624,4484.0,78.0,0.625,0.517


CPU times: user 36.5 ms, sys: 714 μs, total: 37.2 ms
Wall time: 106 ms


Get the combined training+validation data split

In [12]:
%%time
df_train_val = pd.concat([df_train, df_val], ignore_index=True)
print(f"Obtained {len(df_train_val):,} rows of training+validation data")
with pd.option_context('display.max_columns', None):
    display(df_train_val.head())

Obtained 7,978 rows of training+validation data


,clientnum,is_churned,customer_age,gender,dependent_count,education_level,marital_status,income_category,card_category,months_on_book,num_products,months_inactive_12_mon,contacts_count_12_mon,credit_limit,total_revolv_bal,avg_open_to_buy,total_amt_chng_q4_q1,total_trans_amt,total_trans_ct,total_ct_chng_q4_q1,avg_utilization_ratio
0,714283458,0,40.0,M,2.0,College,Single,$80K - $120K,Blue,36.0,5.0,1.0,4.0,14544.0,0.0,14544.0,0.768,4064.0,92.0,0.769,0.000
1,787587033,0,42.0,F,4.0,Graduate,Single,Less than $40K,Blue,32.0,3.0,1.0,2.0,2996.0,1992.0,1004.0,0.948,4463.0,87.0,0.740,0.665
2,714672933,0,52.0,F,3.0,Graduate,Married,$40K - $60K,Blue,36.0,6.0,4.0,2.0,3143.0,2268.0,875.0,0.801,4417.0,84.0,0.680,0.722
3,714974658,0,48.0,F,4.0,College,Married,Less than $40K,Blue,36.0,6.0,1.0,4.0,2464.0,1867.0,597.0,0.600,1219.0,35.0,1.333,0.758
4,712049208,0,56.0,M,3.0,Post-Graduate,Married,$60K - $80K,Blue,39.0,3.0,1.0,2.0,3955.0,2517.0,1438.0,0.484,1238.0,25.0,1.083,0.636


CPU times: user 14.7 ms, sys: 1.87 ms, total: 16.5 ms
Wall time: 15.8 ms


Load the test data

In [13]:
%%time
df_test = (
    r2io.pandas_read_parquet_r2(s3_client, bucket_name, r2_key_test)
    .astype({c: float for c in numeric_features})
    .astype({c: str for c in categorical_features+ordinal_features})
)
print(f"Loaded {len(df_test):,} rows of test data")
with pd.option_context('display.max_columns', None):
    display(df_test)

Loaded 2,149 rows of test data


,clientnum,is_churned,customer_age,gender,dependent_count,education_level,marital_status,income_category,card_category,months_on_book,num_products,months_inactive_12_mon,contacts_count_12_mon,credit_limit,total_revolv_bal,avg_open_to_buy,total_amt_chng_q4_q1,total_trans_amt,total_trans_ct,total_ct_chng_q4_q1,avg_utilization_ratio
0,708223383,1,47.0,F,3.0,High School,Married,Unknown,Blue,39.0,6.0,3.0,4.0,11410.0,979.0,10431.0,1.049,2736.0,38.0,0.462,0.086
1,715052583,0,45.0,M,3.0,Doctorate,Single,$60K - $80K,Silver,34.0,3.0,3.0,1.0,27494.0,879.0,26615.0,0.671,14375.0,112.0,0.672,0.032
2,719910333,0,43.0,F,3.0,Unknown,Unknown,Less than $40K,Blue,36.0,4.0,4.0,2.0,5853.0,1190.0,4663.0,0.936,3595.0,80.0,0.667,0.203
3,785328408,0,37.0,F,4.0,Graduate,Married,Less than $40K,Blue,31.0,5.0,2.0,3.0,1758.0,1180.0,578.0,0.744,2013.0,41.0,0.367,0.671
4,716321583,0,50.0,F,3.0,Unknown,Single,$40K - $60K,Blue,36.0,5.0,3.0,3.0,12740.0,1173.0,11567.0,1.040,3441.0,55.0,0.719,0.092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2144,817247658,0,45.0,M,3.0,High School,Unknown,$60K - $80K,Blue,27.0,6.0,2.0,3.0,21317.0,0.0,21317.0,0.833,3814.0,67.0,1.030,0.000
2145,816834933,0,34.0,F,4.0,Graduate,Single,Unknown,Silver,29.0,6.0,2.0,2.0,30702.0,673.0,30029.0,0.721,2652.0,64.0,0.524,0.022
2146,790221633,0,44.0,F,4.0,Graduate,Single,$40K - $60K,Blue,38.0,3.0,3.0,3.0,3897.0,2296.0,1601.0,0.746,4627.0,77.0,0.791,0.589
2147,719535558,0,47.0,M,4.0,College,Divorced,$40K - $60K,Blue,36.0,5.0,1.0,1.0,5570.0,1858.0,3712.0,0.783,4129.0,72.0,0.636,0.334


CPU times: user 28.7 ms, sys: 5.84 ms, total: 34.5 ms
Wall time: 102 ms


### Data Schema

Define the data schema using the lists of features defined earlier

In [14]:
schema = DataDefinition(
    id_column="clientnum",
    numerical_columns=numeric_features,
    categorical_columns=ordinal_features + categorical_features,
)

### Datasets

#### Validation

Define datasets from the loaded train and validation data splits

In [15]:
# train
dataset_train = Dataset.from_pandas(
    pd.DataFrame(df_train), data_definition=schema
)

# validation
dataset_val = Dataset.from_pandas(pd.DataFrame(df_val), data_definition=schema)

The training data is the *reference* data and the validation data is the *current* data.

#### Evaluation

Define datasets from the loaded train+validation and test data splits

In [16]:
# combined train+validation
dataset_train_val = Dataset.from_pandas(
    pd.DataFrame(df_train_val), data_definition=schema
)

# test
dataset_test = Dataset.from_pandas(
    pd.DataFrame(df_test), data_definition=schema
)

The combined *training+validation* data is the *reference* data and the *test* data is the *current* data.

## Test Data Drift

Define the metrics to be checked for drift per column

In [17]:
metrics = [
    em.ValueDrift(column=c, method="wasserstein", threshold=0.05)
    for c in numeric_features
] + [
    em.ValueDrift(column=c, method="jensenshannon", threshold=0.05)
    for c in ordinal_features + categorical_features
]

### Validation

Check for data drift in the validation data relative to the training data

In [18]:
%%time
df_summary = (
    run_column_data_tests(metrics, dataset_val, dataset_train)
    .assign(
        status_manual=lambda df: (
            (df["value"] > df["threshold"]).map(
                {True: "FAIL", False: "SUCCESS"}
            )
        )
    )
)
display(
    df_summary['status']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
    .merge(
        (
            df_summary['status_manual']
            .value_counts(normalize=True)
            .mul(100)
            .reset_index()
        ),
        left_on=['status'],
        right_on=['status_manual'],
        suffixes=('', '_manual'),
    )
    .drop(columns=['status_manual'])
)
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(
        df_summary.drop(columns=["id"])
        .query("status == 'FAIL'")
        .style.set_caption("Failing tests are shown below")
    )

,status,proportion,proportion_manual
0,SUCCESS,88.235294,88.235294
1,FAIL,11.764706,11.764706


,column,metric_name,value,threshold,condition,category,is_count,description,tolerance,status,status_manual
0,customer_age,ValueDrift,0.053467,0.050000,drift,[],True,Drift score is 0.05. The drift detection method is Wasserstein distance (normed). The drift threshold is 0.05.,nan,FAIL,FAIL
2,months_on_book,ValueDrift,0.052457,0.050000,drift,[],True,Drift score is 0.05. The drift detection method is Wasserstein distance (normed). The drift threshold is 0.05.,nan,FAIL,FAIL


CPU times: user 876 ms, sys: 14.3 ms, total: 891 ms
Wall time: 890 ms


**Notes**

Below are the columns shown above

1. `column`
   - name of feature
2. `metric_name`
   - name of metric used to check the data (for drift, this is `ValueDrift`)
3. `value`
   - this is the p-value of the statistical test used to check for drift
4. `threshold`
   - this is the p-value threshold against which to compare `value` when determining if drift is present
5. `condition`
   - `drift`
   - `le` (less than or equal to)
   - `eq` (equal to)
6. `category`
   - name of category tested
   - for numerical columns, this is an empty list
7. `is_count`
   - whether the `value` is a absolute (`True`) or relative (`False`) value
8. `description`
   - description of test
9. `tolerance`
   - if the `value` if a relative value, then this is the acceptable tolerance in the `value` of the reference dataset; the `value` from the reference dataset should fall within this tolerance band
10. `status`
    - outcome of test (`SUCCESS` or `FAIL`)
11. `status_manual`
    - manually verified outcome of test (`SUCCESS` or `FAIL`)

### Evaluation

Check for data drift in the test data relative to the combined training+validation data

In [19]:
%%time
df_summary = (
    run_column_data_tests(metrics, dataset_test, dataset_train_val)
    .assign(
        status_manual=lambda df: (
            (df["value"] > df["threshold"]).map(
                {True: "FAIL", False: "SUCCESS"}
            )
        )
    )
)
display(
    df_summary['status']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
    .merge(
        (
            df_summary['status_manual']
            .value_counts(normalize=True)
            .mul(100)
            .reset_index()
        ),
        left_on=['status'],
        right_on=['status_manual'],
        suffixes=('', '_manual'),
    )
    .drop(columns=['status_manual'])
)
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(
        df_summary.drop(columns=["id"])
        .query("status == 'FAIL'")
        .style.set_caption("Failing tests are shown below")
    )

,status,proportion,proportion_manual
0,SUCCESS,94.117647,94.117647
1,FAIL,5.882353,5.882353


,column,metric_name,value,threshold,condition,category,is_count,description,tolerance,status,status_manual
5,contacts_count_12_mon,ValueDrift,0.067341,0.050000,drift,[],True,Drift score is 0.07. The drift detection method is Wasserstein distance (normed). The drift threshold is 0.05.,nan,FAIL,FAIL


CPU times: user 8.58 s, sys: 8.68 ms, total: 8.59 s
Wall time: 797 ms


### Observations

Excluding the target (`is_churned`) and customer identifier (`clientnum`) columns and considering the other 19 columns, it is reassuring that drift is observed in

1. two columns in the unseen (validation) data relative to the reference (training data)
2. one column in the unseen (test) data relative to the reference (combined train+validation data)

This suggests one of the following is true for the impacted columns

1. outliers might be present in the unseen data relative to the reference data
2. there is a systematic change that has caused the distribution of the features to shift in the unseen data relative to the reference data

## Test Data Quality

### Dataset Level

Define metrics to be used to test data quality at the dataset level

In [20]:
metrics = [
    em.ConstantColumnsCount(),
    em.EmptyRowsCount(),
    em.EmptyColumnsCount(),
    em.DuplicatedRowCount(),
    em.DuplicatedColumnsCount(),
    em.DatasetMissingValueCount(),
    em.AlmostConstantColumnsCount(),
]

#### Validation

Check for data quality issues at the dataset level in the validation data relative to the training data

In [21]:
%%time
df_summary = run_dataset_data_tests(metrics, dataset_val, dataset_train)
with pd.option_context('display.max_colwidth', None):
    display(df_summary.drop(columns=['id']))
(
    df_summary['status']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)

,metric_name,value,condition,description,status
0,ConstantColumnsCount,0.0,le,Count of constant columns in dataset: Actual value 0.000 >= 0.000 ± 0.000,SUCCESS
1,EmptyRowsCount,0.0,eq,Count of empty rows in dataset: Actual value 0.000 expected 0.000 ± 0.000,SUCCESS
2,EmptyColumnsCount,0.0,le,Count of empty columns in dataset: Actual value 0.000 >= 0.000 ± 0.000,SUCCESS
3,DuplicatedRowCount,0.0,eq,Duplicated row count in dataset: Actual value 0.000 expected 0.000 ± 0.000,SUCCESS
4,DuplicatedColumnsCount,0.0,le,Duplicated column count in dataset: Actual value 0.000 >= 0.000 ± 0.000,SUCCESS
5,DatasetMissingValueCount,0.0,eq,Count and share of missing values in dataset: Actual value 0.000 expected 0.000 ± 0.000,SUCCESS
6,AlmostConstantColumnsCount,0.0,le,Almost constant column count in dataset (eps=0.95): Actual value 0.000 >= 0.000 ± 0.000,SUCCESS


CPU times: user 107 ms, sys: 18 μs, total: 107 ms
Wall time: 107 ms


,status,proportion
0,SUCCESS,100.0


#### Evaluation

Check for data quality issues at the dataset level in the test data relative to the combined training+validation data

In [22]:
%%time
df_summary = run_dataset_data_tests(metrics, dataset_test, dataset_train_val)
with pd.option_context('display.max_colwidth', None):
    display(df_summary.drop(columns=['id']))
(
    df_summary['status']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)

,metric_name,value,condition,description,status
0,ConstantColumnsCount,0.0,le,Count of constant columns in dataset: Actual value 0.000 >= 0.000 ± 0.000,SUCCESS
1,EmptyRowsCount,0.0,eq,Count of empty rows in dataset: Actual value 0.000 expected 0.000 ± 0.000,SUCCESS
2,EmptyColumnsCount,0.0,le,Count of empty columns in dataset: Actual value 0.000 >= 0.000 ± 0.000,SUCCESS
3,DuplicatedRowCount,0.0,eq,Duplicated row count in dataset: Actual value 0.000 expected 0.000 ± 0.000,SUCCESS
4,DuplicatedColumnsCount,0.0,le,Duplicated column count in dataset: Actual value 0.000 >= 0.000 ± 0.000,SUCCESS
5,DatasetMissingValueCount,0.0,eq,Count and share of missing values in dataset: Actual value 0.000 expected 0.000 ± 0.000,SUCCESS
6,AlmostConstantColumnsCount,0.0,le,Almost constant column count in dataset (eps=0.95): Actual value 0.000 >= 0.000 ± 0.000,SUCCESS


CPU times: user 105 ms, sys: 1.01 ms, total: 106 ms
Wall time: 105 ms


,status,proportion
0,SUCCESS,100.0


#### Observations

At the dataset level, there are no data quality issues in the unseen data (validation or test data split) relative to the reference data (training or combined train+validation data split).

### Column Level

#### Validation

Define the metrics to be checked for data quality at the column level, using the training data to get reference values

In [23]:
metrics = (
    [em.MissingValueCount(column=c) for c in [numeric_features[0]]]
    + [
        em.OutListValueCount(column=c, values=df_train[c].unique().tolist())
        for c in categorical_features
    ]
    + [
        em.OutListValueCount(column=c, values=df_train[c].unique().tolist())
        for c in ordinal_features
    ]
)

Test for data quality issues per column in the validation data relative to the training data

In [24]:
%%time
df_summary = (
    run_column_data_tests(metrics, dataset_val, dataset_train)
    .pipe(extract_expected_bounds)
    .assign(
        status_manual=lambda df: (
            (df['actual'] >= df['expected_min'])
            & (df['actual'] <= df['expected_max'])
        ).map({True: "SUCCESS", False: "FAIL"})
    )
)
display(
    df_summary['status']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
    .merge(
        (
            df_summary['status_manual']
            .value_counts(normalize=True)
            .mul(100)
            .reset_index()
        ),
        left_on=['status'],
        right_on=['status_manual'],
        suffixes=('', '_manual'),
    )
    .drop(columns=['status_manual'])
)
if not df_summary.query("status == 'FAIL'").empty:
    with pd.option_context(
        "display.max_colwidth", None, "display.max_rows", None
    ):
        display(
            df_summary
            .drop(columns=["id"])
            .query("status == 'FAIL'")
            .style
            .set_caption('Failing tests are shown below')
        )
else:
    with pd.option_context(
        "display.max_colwidth", None, "display.max_rows", None
    ):
        display(
            df_summary
            .dropna(axis=1, how='all')
            .drop(columns=["id"])
            .style
            .set_caption("No failing tests. All test outcomes shown below.")
        )

,status,proportion,proportion_manual
0,SUCCESS,100.0,100.0


,column,metric_name,value,condition,category,is_count,description,tolerance,status,actual,expected_min,expected,expected_max,status_manual
0,customer_age,MissingValueCount,0.000000,eq,[],True,Column 'customer_age' missing values: Actual value 0.000 expected 0.000 ± 0.000,0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
1,card_category,OutListValueCount,0.000000,eq,[],True,"Column 'card_category' values out of list [Blue, Silver, Gold, Platinum]: Actual value 0.000 expected 0.000 ± 0.000",0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
2,marital_status,OutListValueCount,0.000000,eq,[],True,"Column 'marital_status' values out of list [Single, Married, Divorced, Unknown]: Actual value 0.000 expected 0.000 ± 0.000",0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
3,income_category,OutListValueCount,0.000000,eq,[],True,"Column 'income_category' values out of list [$80K - $120K, Less than $40K, $40K - $60K, $60K - $80K, $120K +, Unknown]: Actual value 0.000 expected 0.000 ± 0.000",0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
4,education_level,OutListValueCount,0.000000,eq,[],True,"Column 'education_level' values out of list [College, Graduate, Post-Graduate, Uneducated, High School, Doctorate, Unknown]: Actual value 0.000 expected 0.000 ± 0.000",0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS


CPU times: user 37.2 ms, sys: 41 μs, total: 37.2 ms
Wall time: 36.2 ms


**Notes**

Below are the three columns shown here that were not seen before

1. `actual`
   - actual value for test against a feature, in the current dataset
2. `expected_min` 
   - minimum allowed value in current dataset
3. `expected_max` 
   - maximum allowed value in current dataset

All three columns are used to manually determine the test outcome in `status_manual`.

#### Evaluation

Define the metrics to be checked for data quality at the column level using the combined training+validation data to get reference values

In [25]:
metrics = (
    # test that same number of missing values are in current and reference data
    [em.MissingValueCount(column=c) for c in numeric_features]
    # test that same number of categorical sub-categories are found in current
    # and reference datasets
    + [
        em.OutListValueCount(
            column=c, values=df_train_val[c].unique().tolist()
        )
        for c in categorical_features
    ]
    # test that same number of ordinal sub-categories are found in current and
    # reference datasets
    + [
        em.OutListValueCount(
            column=c, values=df_train_val[c].unique().tolist()
        )
        for c in ordinal_features
    ]
)

Test for data quality issues per column in the test data relative to the combined validation+training data

In [26]:
%%time
df_summary = (
    run_column_data_tests(metrics, dataset_test, dataset_train_val)
    .pipe(extract_expected_bounds)
    .assign(
        status_manual=lambda df: (
            (df['actual'] >= df['expected_min'])
            & (df['actual'] <= df['expected_max'])
        ).map({True: "SUCCESS", False: "FAIL"})
    )
)
display(
    df_summary['status']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
    .merge(
        (
            df_summary['status_manual']
            .value_counts(normalize=True)
            .mul(100)
            .reset_index()
        ),
        left_on=['status'],
        right_on=['status_manual'],
        suffixes=('', '_manual'),
    )
    .drop(columns=['status_manual'])
)
if not df_summary.query("status == 'FAIL'").empty:
    with pd.option_context(
        "display.max_colwidth", None, "display.max_rows", None
    ):
        display(
            df_summary
            .drop(columns=["id"])
            .query("status == 'FAIL'")
            .style
            .set_caption('Failing tests are shown below')
        )
else:
    with pd.option_context(
        "display.max_colwidth", None, "display.max_rows", None
    ):
        display(
            df_summary
            .dropna(axis=1, how='all')
            .drop(columns=["id"])
            .style
            .set_caption("No failing tests. All test outcomes shown below.")
        )

,status,proportion,proportion_manual
0,SUCCESS,100.0,100.0


,column,metric_name,value,condition,category,is_count,description,tolerance,status,actual,expected_min,expected,expected_max,status_manual
0,customer_age,MissingValueCount,0.000000,eq,[],True,Column 'customer_age' missing values: Actual value 0.000 expected 0.000 ± 0.000,0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
1,dependent_count,MissingValueCount,0.000000,eq,[],True,Column 'dependent_count' missing values: Actual value 0.000 expected 0.000 ± 0.000,0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
2,months_on_book,MissingValueCount,0.000000,eq,[],True,Column 'months_on_book' missing values: Actual value 0.000 expected 0.000 ± 0.000,0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
3,num_products,MissingValueCount,0.000000,eq,[],True,Column 'num_products' missing values: Actual value 0.000 expected 0.000 ± 0.000,0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
4,months_inactive_12_mon,MissingValueCount,0.000000,eq,[],True,Column 'months_inactive_12_mon' missing values: Actual value 0.000 expected 0.000 ± 0.000,0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
5,contacts_count_12_mon,MissingValueCount,0.000000,eq,[],True,Column 'contacts_count_12_mon' missing values: Actual value 0.000 expected 0.000 ± 0.000,0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
6,total_revolv_bal,MissingValueCount,0.000000,eq,[],True,Column 'total_revolv_bal' missing values: Actual value 0.000 expected 0.000 ± 0.000,0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
7,avg_open_to_buy,MissingValueCount,0.000000,eq,[],True,Column 'avg_open_to_buy' missing values: Actual value 0.000 expected 0.000 ± 0.000,0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
8,total_amt_chng_q4_q1,MissingValueCount,0.000000,eq,[],True,Column 'total_amt_chng_q4_q1' missing values: Actual value 0.000 expected 0.000 ± 0.000,0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS
9,total_trans_amt,MissingValueCount,0.000000,eq,[],True,Column 'total_trans_amt' missing values: Actual value 0.000 expected 0.000 ± 0.000,0.100000,SUCCESS,0.000000,0.000000,0.000000,0.000000,SUCCESS


CPU times: user 48 ms, sys: 39 μs, total: 48.1 ms
Wall time: 47.1 ms


#### Observations

In terms of missing values in numerical columns or unexpected categorical values, there are no data quality issues at the column level during model validation or evaluation.

## Test Data Descriptive Statistics

### Column Level

Define the numerical metrics to be checked for descriptive statistics at the column level

In [27]:
metrics_numerical = (
    [em.MinValue(column=c) for c in numeric_features]
    + [em.MeanValue(column=c) for c in numeric_features]
    + [em.MedianValue(column=c) for c in numeric_features]
    + [em.StdValue(column=c) for c in numeric_features]
    + [em.MaxValue(column=c) for c in numeric_features]
)

These tests are to check that the minimum, median, mean, standard deviation and maximum values in current data are within 10% of the corresponding values in the reference data.

#### Validation

Define the metrics to be checked for descriptive statistics at the column level, using the training data to get reference values

In [28]:
metrics = (
    metrics_numerical
    # test that same frequency of ordinal sub-categories are found in current
    # and reference datasets
    + [
        em.CategoryCount(column=c, category=ca)
        for c in ordinal_features
        for ca in df_train[c].unique().tolist()
    ]
    # test that same frequency of categorical sub-categories are found in
    # current and reference datasets
    + [
        em.CategoryCount(column=c, category=ca)
        for c in categorical_features
        for ca in df_train[c].unique().tolist()
    ]
)

Test for data statistics issues per column in the validation data relative to the training data

In [29]:
%%time
df_summary = (
    run_column_data_tests(metrics, dataset_val, dataset_train)
    .pipe(extract_expected_bounds)
    .assign(
        status_manual=lambda df: (
            (df['actual'] >= df['expected_min'])
            & (df['actual'] <= df['expected_max'])
        ).map({True: "SUCCESS", False: "FAIL"})
    )
)
display(
    df_summary['status']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
    .merge(
        (
            df_summary['status_manual']
            .value_counts(normalize=True)
            .mul(100)
            .reset_index()
        ),
        left_on=['status'],
        right_on=['status_manual'],
        suffixes=('', '_manual'),
    )
    .drop(columns=['status_manual'])
)
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(
        df_summary
        .drop(columns=["id"])
        .query("(status == 'FAIL') | (status_manual == 'FAIL')")
        .style
        .set_caption(
            'Mismatch between built-in and manual test status is highlighted '
            'in red'
        )
        .apply(
            lambda row: (
                [
                    'background-color: darkred; color: white; font-weight: bold'
                ] * len(row)
                if (
                    (row['status'] == 'FAIL')
                    & (row['status_manual'] == 'SUCCESS')
                )
                else [''] * len(row)
            ),
            axis=1
        )
    )

,status,proportion,proportion_manual
0,SUCCESS,89.534884,90.697674
1,FAIL,10.465116,9.302326


,column,metric_name,value,threshold,condition,category,is_count,description,tolerance,status,actual,expected_min,expected,expected_max,status_manual
7,avg_open_to_buy,MinValue,15.000000,nan,eq,[],True,"Minimal value of 'avg_open_to_buy': Actual value 15.000 , but expected 3.000 ± 0.300",0.100000,FAIL,15.000000,2.700000,3.000000,3.300000,FAIL
31,contacts_count_12_mon,MedianValue,3.000000,nan,eq,[],True,"Median value of 'contacts_count_12_mon': Actual value 3.000 , but expected 2.000 ± 0.200",0.100000,FAIL,3.000000,1.800000,2.000000,2.200000,FAIL
60,total_amt_chng_q4_q1,MaxValue,2.145000,nan,eq,[],True,"Maximum value of 'total_amt_chng_q4_q1': Actual value 2.145 , but expected 3.355 ± 0.336",0.100000,FAIL,2.145000,3.019000,3.355000,3.691000,FAIL
63,total_ct_chng_q4_q1,MaxValue,2.500000,nan,eq,[],True,"Maximum value of 'total_ct_chng_q4_q1': Actual value 2.500 , but expected 3.571 ± 0.357",0.100000,FAIL,2.500000,3.214000,3.571000,3.928000,FAIL
69,income_category,CategoryCount,137.000000,nan,eq,['$120K +'],False,"Column 'income_category' categories '$120K +': Actual value 0.081 , but expected 0.067 ± 0.007",0.100000,FAIL,0.081000,0.060000,0.067000,0.074000,FAIL
71,education_level,CategoryCount,152.000000,nan,eq,['College'],False,"Column 'education_level' categories 'College': Actual value 0.090 , but expected 0.103 ± 0.010",0.100000,FAIL,0.090000,0.093000,0.103000,0.113000,FAIL
76,education_level,CategoryCount,89.000000,nan,eq,['Doctorate'],False,"Column 'education_level' categories 'Doctorate': Actual value 0.053 , but expected 0.042 ± 0.004",0.100000,FAIL,0.053000,0.038000,0.042000,0.046000,FAIL
81,card_category,CategoryCount,4.000000,nan,eq,['Platinum'],False,"Column 'card_category' categories 'Platinum': Actual value 0.002 , but expected 0.002 ± 0.000",0.100000,FAIL,0.002000,0.002000,0.002000,0.002000,SUCCESS
85,marital_status,CategoryCount,110.000000,nan,eq,['Unknown'],False,"Column 'marital_status' categories 'Unknown': Actual value 0.065 , but expected 0.077 ± 0.008",0.100000,FAIL,0.065000,0.069000,0.077000,0.085000,FAIL


CPU times: user 909 ms, sys: 2.12 ms, total: 911 ms
Wall time: 911 ms


At the column level, it is reassuring at most four out of 13 numerical features columns are failing stats tests. Three out of the four failing tests are due to values in the unseen data falling above the maximum value or below the minimum value in the reference data, including a 10% margin.

The tests on the non-numerical features were to check if the proportion of values for a single category in each feature was the same between the unseen and reference data, including a 10% margin. Four out of the 21 unique categories across the ordinal and categorical features combined are failing tests (excluding the <font color='darkred'>**highlighted row**</font> which was incorrectly labeled as failing, as shown by the manual status check in `status_manual`).

Failing tests in the numerical and non-numerical columns features suggests there are outliers in the unseen data relative to the reference data. All the failing tests for the categorical and ordinal features involve values that are close to the expected range. However, the `avg_open_to_buy` numerical feature contains at least one row (customer) with a minimum value well outside the expected range. We can manually verify this, as shown below

In [30]:
c = "avg_open_to_buy"
df_stats = (
    df_val[c]
    .describe()
    .rename("validation")
    .to_frame()
    .merge(
        df_train[c].describe().rename("train").to_frame(),
        left_index=True,
        right_index=True,
        how="left",
    )
    .assign(column=c)
)
display(
    df_stats.style.apply(
        lambda x: [
            "background-color: yellow" if x.name in ["min"] else "" for i in x
        ],
        axis=1,
    )
)

,validation,train,column
count,1693.000000,6285.000000,avg_open_to_buy
mean,7696.624867,7342.160748,avg_open_to_buy
std,9283.119865,8966.590661,avg_open_to_buy
min,15.000000,3.000000,avg_open_to_buy
25%,1295.000000,1331.000000,avg_open_to_buy
50%,3703.000000,3418.000000,avg_open_to_buy
75%,10524.000000,9627.000000,avg_open_to_buy
max,34516.000000,34516.000000,avg_open_to_buy


This could indicate a systematic change in customer behaviour characterized by this one feature `avg_open_to_buy`. However, the tests for the median, mean and standard deviation values for this column passed and the data drift test also passed. So, this suggests the failing test is due to outliers in this column rather than am underlying change in customer behaviour between the train and validation data splits. This should be verified during exploratory data analysis.

#### Evaluation

Define the metrics to be checked for descriptive statistics at the column level, using the combined train+validation data to get reference values

In [31]:
metrics = (
    metrics_numerical
    + [
        em.CategoryCount(column=c, category=ca)
        for c in ordinal_features
        for ca in df_train_val[c].unique().tolist()
    ]
    + [
        em.CategoryCount(column=c, category=ca)
        for c in categorical_features
        for ca in df_train_val[c].unique().tolist()
    ]
)

Test for data statistics issues per column in the test data relative to the combined train+validation data

In [32]:
%%time
df_summary = (
    run_column_data_tests(metrics, dataset_test, dataset_train_val)
    .pipe(extract_expected_bounds)
    .assign(
        status_manual=lambda df: (
            (df['actual'] >= df['expected_min'])
            & (df['actual'] <= df['expected_max'])
        ).map({True: "SUCCESS", False: "FAIL"})
    )
)
display(
    df_summary['status']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
    .merge(
        (
            df_summary['status_manual']
            .value_counts(normalize=True)
            .mul(100)
            .reset_index()
        ),
        left_on=['status'],
        right_on=['status_manual'],
        suffixes=('', '_manual'),
    )
    .drop(columns=['status_manual'])
)
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(
        df_summary
        .drop(columns=["id"])
        .query("(status == 'FAIL') | (status_manual == 'FAIL')")
        .style
        .set_caption('Failing tests are shown below')
    )

,status,proportion,proportion_manual
0,SUCCESS,94.186047,94.186047
1,FAIL,5.813953,5.813953


,column,metric_name,value,threshold,condition,category,is_count,description,tolerance,status,actual,expected_min,expected,expected_max,status_manual
7,avg_open_to_buy,MinValue,14.000000,nan,eq,[],True,"Minimal value of 'avg_open_to_buy': Actual value 14.000 , but expected 3.000 ± 0.300",0.100000,FAIL,14.000000,2.700000,3.000000,3.300000,FAIL
8,total_amt_chng_q4_q1,MinValue,0.010000,nan,eq,[],True,"Minimal value of 'total_amt_chng_q4_q1': Actual value 0.010 , but expected 0.000 ± 0.000",0.100000,FAIL,0.010000,0.000000,0.000000,0.000000,FAIL
9,total_trans_amt,MinValue,596.000000,nan,eq,[],True,"Minimal value of 'total_trans_amt': Actual value 596.000 , but expected 510.000 ± 51.000",0.100000,FAIL,596.000000,459.000000,510.000000,561.000000,FAIL
69,income_category,CategoryCount,168.000000,nan,eq,['$120K +'],False,"Column 'income_category' categories '$120K +': Actual value 0.078 , but expected 0.070 ± 0.007",0.100000,FAIL,0.078000,0.063000,0.070000,0.077000,FAIL
73,education_level,CategoryCount,78.000000,nan,eq,['Post-Graduate'],False,"Column 'education_level' categories 'Post-Graduate': Actual value 0.036 , but expected 0.055 ± 0.005",0.100000,FAIL,0.036000,0.050000,0.055000,0.060000,FAIL


CPU times: user 1.01 s, sys: 1.15 ms, total: 1.01 s
Wall time: 1.01 s


There are failing tests for extreme values (minimum or maximum), like we saw during model validation above. Again, it is reassuring that only three numerical features have failing tests and there is only one failing test for each of these features. As we saw in the previous section `avg_open_to_buy` likely has outliers.

None of the 13 categories in the ordinal features have failing tests.

In the categorical features, two of the eight categories have a failing test. As we saw in the previous seciton, although the actual proportions are failing, they are close the expected range.

## Limitations

We have assumed that during model validation and evaluation the current and reference datasets are available at the time of running these tests. This is acceptable since we are not deploying a model to production as part of this project. These tests are only being used to inform exploratory data analysis and to help interpret model performance if features with failing tests are used during model development.

## Conclusions

Overall, the diagnostics suggest that the data used for validation and testing is largely consistent with the training distribution, with no meaningful data quality concerns and only limited, localized signs of drift. The small number of drifted features and failing statistical tests are primarily driven by extreme values. These indicate that outliers, rather than systematic shifts in customer behavior, are the most likely cause of discrepancies between reference and unseen datasets. This is further supported by stable aggregate statistics (mean, median, standard deviation) and minimal deviation in categorical proportions.

As a result, the model evaluation can be considered reliable. Targeted exploratory data analysis on features like `avg_open_to_buy` is warranted to confirm the presence and impact of outliers before interpreting churn probabilities or making downstream business decisions.